In [1]:
!pip install tf-models-official

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [2]:
!pip install pipreqs

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [3]:
!pip install wandb --upgrade

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [4]:
import wandb
# wandb.login()
# !wandb login --relogin
wandb.login()

wandb: Currently logged in as: fleur. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
import gc
import numpy as np
import pandas as pd
import cv2

# import splitfolders
import h5py
from matplotlib import pyplot as plt
%matplotlib inline
from matplotlib import rcParams
import seaborn as sns
from PIL import Image
from imageio import imwrite
from tqdm import tqdm
import seaborn as sns


import tensorflow as tf
import tensorflow_models as tfm
import tensorflow_probability as tfp

from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.utils import plot_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.applications import (
    InceptionResNetV2,
    ResNet50,
    InceptionV3,
    DenseNet121,
)
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D, LeakyReLU, BatchNormalization
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, Callback, CSVLogger
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.metrics import Accuracy, Precision, Recall, AUC, TopKCategoricalAccuracy, TruePositives, FalsePositives, TrueNegatives, FalseNegatives
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint, WandbCallback
 
# !pipreqs
print("Imported")


Imported


In [6]:
try:
    from google.colab import drive
    drive.mount("/content/drive/", force_remount=True)
    google_drive_prefix = "/content/drive/My Drive"
    data_prefix = "{}/brain/".format(google_drive_prefix)
except ModuleNotFoundError: 
    data_prefix = "data/"

Mounted at /content/drive/


In [7]:


# path = '/kaggle/input/cropped-images/Cropped_images'
path = '/content/drive/My Drive/Datasets/Cropped_images/'

PREFIX = "inat" # convenient for tracking local data
# model_dir ="/content/drive/My Drive/Models/RadImageNet-DenseNet121_notop.h5"
# model_dir ="/content/drive/My Drive/Models/RadImageNet-ResNet50_notop.h5"

model_dir = '/content/drive/My Drive/Models/RadImageNet-IRV2_notop.h5'
# model_dir = '/kaggle/input/radimagenet-irv2/RadImageNet-IRV2_notop.h5'


# Meningioma_generator_dir = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Code_v02/Models/Menin_RGB_generator_model_4000.h5"
Meningioma_generator_dir = '/content/drive/My Drive/Models/Menin_RGB_generator_model_4000.h5' 
Pituitary_generator_dir = '/content/drive/My Drive/Models/Pituitary_RGB_generator_3000_0.5_drop.h5'
# Pituitary_generator_dir = '/kaggle/input/pit-rgb-generator-3000/generator_model_3000.h5'
# results = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Code_v02/Images/"
CODE_OUTPUTS = '/content/drive/My Drive/Code_Outputs/'
csv_logger = CSVLogger(f'{CODE_OUTPUTS}training.log')

In [8]:
# sweep_config = {
#     'method': "random",
#     'metric': {
#         'name': 'loss',
#         'goal': 'minimize',
#     },
#     'parameters': {
#         "nodes": {
#             "values": [128]
#         },
#         "learning_rate": {
#             'values': [0.0001, 0.001, 0.01]
#         },
#         "dropout_1": {
#             'values': [0.2, 0.3]
#         },
#         "dropout_2": {
#             'values': [0.2, 0.3]
#         }
#         "batch_size": {
#             "distribution": "q_log_uniform",
#             "q": 1,
#             "min": 32,
#             "max": 128
#         },
#     },
# }
# sweep_id = wandb.sweep(sweep_config, project="Mermoire_2023_Version_01")
# import pprint

# pprint.pprint(sweep_config)

# Helper Functions

In [9]:
def display_images(images, name):
#     Display 9 Images
    for i in range(9):
        # define subplot
        plt.subplot(3, 3, 1 + i)
        plt.axis("off")
        # plot single image
        image = images[i, :, :, 0]
#         image = images[i]
#         image = np.uint8(image)
        plt.imshow(image, cmap="gray")
        plt.savefig(
            f'{CODE_OUTPUTS}{name}image.jpg',
            dpi=300,
            transparent=True,
        )
    plt.show()
    plt.close()
    
def visualize_distribution(images1, images2, count):
    fig, axs = plt.subplots(ncols=1, nrows=1, figsize=(18, 10))

    images1 = images1[:count]
    sns.distplot(images1, label="Real Images", hist=True, color="#fc0328", ax=axs)
    if images2 != None:
        images2 = images2[:count]
        sns.distplot(images2, label="Generated Images", hist=True, color="#0c06c7", ax=axs)
    axs.legend(loc="upper right", prop={"size": 12})
    # pyplot.savefig(
    #     distribution,
    #     bbox_inches="tight",
    #     transparent=True,
    # )
    plt.show()
    plt.close()

def visualize_class_distribution(classes, name):
    fig, axs = plt.subplots(ncols=1, nrows=1, figsize=(18, 10))
    sns.distplot(classes, label= name, hist=True, color="#fc0328", ax=axs)
    plt.show()
    plt.close()
    

# Variables Setup

In [10]:

LATENT_DIM = 100
DISPLAY_GENRATED_SAMPLES = 10
BATCH_SIZE = 32
IMAGE_SIZE = 128

TRAIN_RATIO = 0.75
VALIDATION_RATIO = 0.20
TEST_RATIO = 0.10

GAN_AUG = True
MIXUP_AUG = False

AUGMENT_GLIOMA = False
AUGMENT_MENINGIOMA = True
AUGMENT_PITUITARY = True

AUGMENTATION_RATE = 50
GLIOMA_AUGMENTATION_NUM= 0
MENINGIOMA_AUGMENTATION_NUM = 4 # 10x50 = 500
PITUITARY_AUGMENTATION_NUM = 3 # 7x50 = 350


# Preprocessing & Enhancement

In [11]:
def Enhancement(image):
    
    NOISE_FLOOR = 15

    def reduce_noise(noisy_img, noise_floor=NOISE_FLOOR):

        clean_img = noisy_img
        clean_img[noisy_img < NOISE_FLOOR] = 0
        return clean_img

    def eualize_darken(image):        

#         img = cv2.imread(image_to_crop_path)    
#         image = reduce_noise(image)
#         print(new_img.shape)

        new_img = cv2.bilateralFilter(image,1,10,10)
#         print(new_img.shape)
        img_hsv = cv2.cvtColor(new_img, cv2.COLOR_RGB2YCrCb)
        img_hsv[:, :, 0] = cv2.equalizeHist(img_hsv[:, :, 0])
        image = cv2.cvtColor(img_hsv, cv2.COLOR_YCrCb2RGB)  
#         print("Equalized!")
        return image
#     image = eualize_darken(image)

    def gammaCorrection(src, gamma):

        invGamma = 1 / gamma
        table = [((i / 255) ** invGamma) * 255 for i in range(256)]
        table = np.array(table, np.uint8)
        image = cv2.LUT((255 * src).astype(np.uint8), table)
        image = (image.astype(np.float)) / 255
#         print("Gamma Corrected!")
        return image
    image = reduce_noise(image)
    image = eualize_darken(image)
    image = gammaCorrection(image, 1.4)
    

#     # Normalize and Scale
#     image = (image - np.min(image)) / (np.max(image) - np.min(image))
#     # Rescale the pixel values to be between -1 and 1
#     image = (image * 2) - 1

    return image

# Import Data

In [15]:
Images = []
Labels = []
Classes = ['glioma', 'meningioma', 'pituitary']
classes_dict = {'glioma': 0, 'meningioma':1, 'pituitary':2}

for Class in classes_dict.keys():
    class_samples = 0
    cpath = os.path.join(path, Class)
    cpath += '/'
    # print(cpath)
    for img in os.listdir(cpath):
        class_samples += 1

        image = cv2.imread(os.path.join(cpath, img), cv2.IMREAD_COLOR)        
        image = cv2.resize(image, (128, 128))
        image = Enhancement(image)
        # Normalize and rescale
        image = (image - np.min(image)) / (np.max(image) - np.min(image))
          # Rescale the pixel values to be between -1 and 1
        image = (image * 2) - 1
        
        Images.append(image)
        Labels.append(classes_dict[Class])
    print(f"There are {class_samples} samples in the class {Class}")


Images = np.array(Images)
Labels = np.array(Labels)

print("Images: ", Images.shape)
print("Labels: ", Labels.shape)



# item, freq = np.array(np.unique(y_train, return_counts=True)).T


# display_images(Images, 'data_import')
# visualize_distribution(Images, None, 20)

<ipython-input-11-6e56ec804611>:32: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  image = (image.astype(np.float)) / 255


There are 1436 samples in the class glioma
There are 708 samples in the class meningioma
There are 930 samples in the class pituitary
Images:  (3074, 128, 128, 3)
Labels:  (3074,)


In [16]:
train_data = np.concatenate((Images, np.expand_dims(Labels, axis=1)), axis=1)

# trainind_data = np.concatenate((Images, Labels), axis=0)
print("train_data: ", train_data.shape)
print("train_data: ", train_data[0])


ValueError: ignored

# Split data 75:15:15

In [13]:
# train is now 75% of the entire data set
x_train, x_test_tmp, y_train, y_test_tmp = train_test_split(Images, Labels, test_size= 1 - TRAIN_RATIO, random_state= 42, shuffle= True)

# test is now 15% of the initial data set
# validation is now 15% of the initial data set
x_val, x_test, y_val, y_test = train_test_split(x_test_tmp, y_test_tmp, test_size=TEST_RATIO/(TEST_RATIO + VALIDATION_RATIO), random_state=42, shuffle= True) 

print("x_train: ", x_train.shape)
print("x_val: ", x_val.shape)
print("x_test: ", x_test.shape)

# Cleaning Memoiry
del Images
del Labels
del x_test_tmp
del y_test_tmp
gc.collect()


x_train:  (2305, 128, 128, 3)
x_val:  (512, 128, 128, 3)
x_test:  (257, 128, 128, 3)


21696

# Generate Images

In [14]:
def GAN_Augmentation(model_dir, label, augmentation_number, augmentation_rate):
    
    generator = load_model(model_dir)
    # generator.summary()
    
    gen_imgs= []
    gen_lbls= []
    generated_labels = []
    interation = 1

    for i in range(augmentation_number): # 7 or 10
#         print(f"Iteration: {interation}")
        noise = tf.random.normal([augmentation_rate, LATENT_DIM])
        GENERATED_IMAGE = generator(noise, training=False)
        generated_images = []
        for i in range(augmentation_rate):
            # print(filename)
            generated = GENERATED_IMAGE[i]
            generated = np.array(generated)
#             print(generated.shape)
#             generated = (generated + 1) / 2
#             generated = generated * (np.max(generated) - np.min(generated)) + np.min(generated)
            generated = ((generated + 1) / 2) * 255.0
            generated = np.uint8(generated)


            generated = Enhancement(generated.astype(np.uint8))
        
            generated_images.append(generated)  
        
        gen_imgs.extend(generated_images)
        interation += 1
    print(f'Generated {len(gen_imgs)} images')
    print("---------------- Process Finished ----------------")
    for i in range(augmentation_number):
        for i in range(augmentation_rate):
            generated_labels.append(label)
    gen_lbls.extend(generated_labels)
#         gen_lbls = np.array(gen_lbls)
    print(f'Generated {len(gen_lbls)} labels')
#     print(f'Labels: {gen_lbls}')
# Display Generated Images
#     display_images(np.array(gen_imgs))
#     visualize_distribution(gen_imgs, None, 20)    
    
    return gen_imgs, gen_lbls

# if AUGMENT_MENINGIOMA == True:
#     print("--------------- Meningioma Augmentation ---------------")
#     generated_images, generated_labels = GAN_Augmentation(Meningioma_generator_dir, 1, AUGMENTATION_RATE, MENINGIOMA_AUGMENTATION_NUM)
#     generated_images = np.array(generated_images)
#     generated_labels = np.array(generated_labels)
#     print("Train: ", x_train.shape)
#     print("generated_images: ", generated_images.shape)
#     augmented_train_images = np.concatenate((x_train, generated_images), axis=0)
#     print("augmented_train_images shape: ", augmented_train_images.shape)

#     print("y_train: ", y_train.shape)
#     print("generated_labels: ", generated_labels.shape)
#     augmented_train_labels = np.concatenate((y_train, generated_labels), axis=0)
#     print("augmented_train_labels shape: ", augmented_train_labels.shape)
#     del x_train
#     del y_train
#     gc.collect()

# if AUGMENT_PITUITARY == True:
#     print("--------------- Pituitary Augmentation ---------------")
#     generated_images, generated_labels = GAN_Augmentation(Pituitary_generator_dir, 2, AUGMENTATION_RATE, PITUITARY_AUGMENTATION_NUM)
#     generated_images = np.array(generated_images)
#     generated_labels = np.array(generated_labels)
#     # print("Train: ", x_train.shape)
#     print("generated_images: ", generated_images.shape)
#     augmented_train_images = np.concatenate((augmented_train_images, generated_images), axis=0)
#     print("augmented_train_images shape: ", augmented_train_images.shape)

#     # print("y_train: ", y_train.shape)
#     print("generated_labels: ", generated_labels.shape)
#     augmented_train_labels = np.concatenate((augmented_train_labels, generated_labels), axis=0)
#     print("augmented_train_labels shape: ", augmented_train_labels.shape)

In [15]:
def MixUp(images, labels):
    
    images = tf.convert_to_tensor(images)
    labels = tf.convert_to_tensor(labels)
    mx = tfm.vision.augment.MixupAndCutmix
    new_img, new_lbl = mx(mixup_alpha= 0.2, num_classes= 3, cutmix_alpha= 0).distort(images, labels)
    mixed_up_images = np.array(new_img)
    mixed_up_labels = np.array(new_lbl)

    display_images(np.array(mixed_up_images), "MixUP")
#     visualize_distribution(augmented_train_images, None, 20) 
    return mixed_up_images, mixed_up_labels

In [16]:
# mixup_images, mixup_labels = MixUp(augmented_train_images, augmented_train_labels)
# print("augmented_train_images: ", augmented_train_images.shape)
# print("augmented_train_labels: ", augmented_train_labels.shape)

In [17]:
# # augmented_train_images = np.array(augmented_train_images, dtype="float") / 255.0
# # x_val = np.array(x_val, dtype="float") / 255.0
# # x_test = np.array(x_test, dtype="float") / 255.0

# sns.distplot(augmented_train_labels,bins="doane",kde=False,hist_kws={"align" : "left"})
# plt.show()

# le = LabelEncoder()
# print("shape: ", augmented_train_labels.shape)
# augmented_train_labels = le.fit_transform(augmented_train_labels)
# augmented_train_labels = to_categorical(augmented_train_labels)

# le = LabelEncoder()
# y_val_enc = le.fit_transform(y_val)
# y_val_enc = to_categorical(y_val)

# print("Train Images Set Before Mixup Augmentation: ", augmented_train_images.shape)
# augmented_train_images_final = np.concatenate((augmented_train_images, mixup_images), axis=0)
# print("Train Images Set After Mixup Augmentation: ", augmented_train_images.shape)

# print("Train Labels Set Before Mixup Augmentation: ", augmented_train_labels.shape)
# print("mixed_up_labels: ", mixup_labels.shape)
# print("mixed_up_labels: ", mixup_labels[0])
# # print("mixed_up_labels: ", mixed_up_labels)
# augmented_train_labels_final = np.concatenate((augmented_train_labels, mixup_labels), axis=0)
# print("Train Labels Set After Mixup Augmentation: ", augmented_train_labels.shape)

# # del x_train
# # del y_train
# del mixup_images
# del mixup_labels
# del augmented_train_images
# del augmented_train_labels
# gc.collect()


In [18]:
print("Class Distribution before augmentation: ")
# visualize_class_distribution(y_train, "Pre_Aug")
np.savetxt(f"{CODE_OUTPUTS}Pre_Aug.csv", y_train, delimiter =", ", fmt ='% s')

def Augmentation_framework(GAN_AUG, MIXUP_AUG, x_train, y_train, y_val):

  if GAN_AUG == True:
    if AUGMENT_MENINGIOMA == True:
      print("--------------- Meningioma Augmentation ---------------")
      generated_images, generated_labels = GAN_Augmentation(Meningioma_generator_dir, 1, AUGMENTATION_RATE, MENINGIOMA_AUGMENTATION_NUM)
      generated_images = np.array(generated_images)
      generated_labels = np.array(generated_labels)

      print(f"Train shape: {x_train.shape}, generated_images shape: {generated_images.shape}")
      Images = np.concatenate((x_train, generated_images), axis=0)
      print("augmented_train_images shape: ", Images.shape)
      print("---------------------------\n")
      
      print(f"y_train shape: {y_train.shape}, generated_labels shape: {generated_labels.shape}")
      Labels = np.concatenate((y_train, generated_labels), axis=0)
      print("augmented_train_labels shape: ", Labels.shape)
      
      del x_train
      del y_train
      gc.collect()

    if AUGMENT_PITUITARY == True:
      print("--------------- Pituitary Augmentation ---------------")
      generated_images, generated_labels = GAN_Augmentation(Pituitary_generator_dir, 2, AUGMENTATION_RATE, PITUITARY_AUGMENTATION_NUM)
      generated_images = np.array(generated_images)
      generated_labels = np.array(generated_labels)
      
      
      print(f"augmented_train_images shape: { Images.shape}, generated_images shape: {generated_images.shape}")
      Images = np.concatenate((Images, generated_images), axis=0) 
      print("augmented_train_images shape: ", Images.shape)

    
      print("---------------------------\n")
      print(f"augmented_train_labels shape: {Labels.shape}, generated_labels shape: {generated_labels.shape}")  
      Labels = np.concatenate((Labels, generated_labels), axis=0)
      print("augmented_train_labels shape: ", Labels.shape)
      
  np.savetxt(f"{CODE_OUTPUTS}GAN_Aug.csv", Labels, delimiter =", ", fmt ='% s')

  if MIXUP_AUG == True:
    mixup_images, mixup_labels = MixUp(Images, Labels)
    print(f"augmented_train_images: {Images.shape}, augmented_train_labels: {Labels.shape}")
  
    np.savetxt(f"{CODE_OUTPUTS}Mixup_Aug.csv", mixup_labels, delimiter =", ", fmt ='% s')
    
    le = LabelEncoder()
    Labels = le.fit_transform(Labels)
    Labels = to_categorical(Labels)


    Images = np.concatenate((Images, mixup_images), axis=0)
    print("Train Images Set After Mixup Augmentation: ", Images.shape)

    Labels = np.concatenate((Labels, mixup_labels), axis=0)
    print("Train Labels Set After Mixup Augmentation: ", Labels.shape)

    le = LabelEncoder()
    y_val = le.fit_transform(y_val)
    y_val = to_categorical(y_val)

    del mixup_images
    del mixup_labels
    gc.collect()
  else:
    le = LabelEncoder()
    Labels = le.fit_transform(Labels)
    Labels = to_categorical(Labels)

    # Images = np.concatenate((Images, mixup_images), axis=0)
    # print("Train Images Set After Mixup Augmentation: ", Images.shape)

    # Labels = np.concatenate((Labels, mixup_labels), axis=0)
    # print("Train Labels Set After Mixup Augmentation: ", Labels.shape)

    le = LabelEncoder()
    y_val = le.fit_transform(y_val)
    y_val = to_categorical(y_val)

  return Images, Labels, y_val

Images, Labels, y_val = Augmentation_framework(GAN_AUG, MIXUP_AUG, x_train, y_train, y_val)
    

Class Distribution before augmentation: 
--------------- Meningioma Augmentation ---------------


<ipython-input-11-6e56ec804611>:32: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  image = (image.astype(np.float)) / 255


Generated 200 images
---------------- Process Finished ----------------
Generated 200 labels
Train shape: (2305, 128, 128, 3), generated_images shape: (200, 128, 128, 3)
augmented_train_images shape:  (2505, 128, 128, 3)
---------------------------

y_train shape: (2305,), generated_labels shape: (200,)
augmented_train_labels shape:  (2505,)
--------------- Pituitary Augmentation ---------------


Generated 150 images
---------------- Process Finished ----------------
Generated 150 labels
augmented_train_images shape: (2505, 128, 128, 3), generated_images shape: (150, 128, 128, 3)
augmented_train_images shape:  (2655, 128, 128, 3)
---------------------------

augmented_train_labels shape: (2505,), generated_labels shape: (150,)
augmented_train_labels shape:  (2655,)


In [19]:
def init_data():
    train_datagen = ImageDataGenerator()
    val_datagen = ImageDataGenerator()
    test_datagen = ImageDataGenerator()

    train_data = train_datagen.flow(Images,
                                     Labels, 
                                     shuffle= True,
                                     seed= 22, 
                                     batch_size= BATCH_SIZE)
    valid_data = val_datagen.flow(x_val, 
                                   y_val, 
                                   shuffle= True,
                                   seed= 22,
                                   batch_size= BATCH_SIZE)
    test_data = test_datagen.flow(x_test,
                                   y_test,
                                   shuffle= True,
                                   seed= 22,
                                   batch_size= BATCH_SIZE)

    return train_data, valid_data, test_data

# train_data, valid_data, test_data = init_data(train_dir=train_dir, valid_dir=valid_dir, test_dir=test_dir)

# val_data, val_labels = valid_data.next()
# print("val_data: ", val_data)
# print("val_labels: ", val_labels.shape)
# def image_wandb_logger(train_data):
#     images, labels=next(train_data)
#     class_dict=train_data.class_indices
#     class_dict_updated = {}

#     for k, v in class_dict.items():
#         class_dict_updated[v]=k

#     print(class_dict_updated)

#     images_list = []
#     labels_list = []

#     for i in range(len(images)):
#         current_image = images[i]
#         current_label = labels[i]
#         current_label = class_dict_updated[np.argmax(current_label)]
#         images_list.append(current_image)
#         labels_list.append(current_label)
#         # cv2.imshow("img", current_image)
#         # print(current_label)
#     return images_list

In [20]:
def train():
    config={
    "dropout_1": 0,
    "dropout_2": 0.1,        
    "activation_1": "relu",
    "activation_2": "softmax",
    "optimizer": "adam",
    "decay_rate": 0.5,
    "loss": "categorical_crossentropy",
    "metric_Acc": "accuracy",
    "epoch": 50,
    "batch_size": 32,
    "nodes": 128,
    "learning_rate": 0.001,
    "layer_num": 1,
    "GAN_Aug": True,
    "Mixup_Aug": False,
    "Classic_Aug": False,
    }
    wandb.init(project="Mermoire_2023_Version_01",config= config)
    config = wandb.config
  
    train_data, valid_data, test_data = init_data()
    n_samples_train = len(train_data)*(32)
    print("n_samples_train: ", n_samples_train)
    n_samples_val = len(valid_data)*(32)
    print("n_samples_val: ", n_samples_val)
    n_samples_test = len(test_data) *(32)
    print("n_samples_test: ", n_samples_test)

#     images_list = image_wandb_logger(train_data)
#     for i in range(len(images_list)):
#       images = wandb.Image(images_list[i], caption="Train Set")
#       wandb.log({"Training": images})
        
    def build_transfer_learning_model(base_model):
        # base_model.summary()
        # tf.keras.utils.plot_model(base_model, to_file=f"{CODE_OUTPUTS}rad_model.png", dpi= 300,  show_shapes=True)

        # `base_model` stands for the pretrained model
        # We want to use the learned weights, and to do so we must freeze them
        for layer in base_model.layers:
            layer.trainable = False
        #     for layer in base_model.layers[:171]:
        #       layer.trainable = False
        #     for layer in base_model.layers[171:]:
        #       layer.trainable = True

        # Declare a sequential model that combines the base model with custom layers
        # def add_hidden_layer(config.layer_num):
        #   for i range(layer_num):
        #     Dense(units=config.nodes, activation= config.activation_1),

        model = tf.keras.Sequential([
            base_model,
            GlobalAveragePooling2D(),
            
#             Dense(units=config.nodes),
#             Dropout(rate=config.dropout_1),
            
            # Dense(units=config.nodes, activation= config.activation_1),
            # BatchNormalization(),
            Dense(units=config.nodes, activation= config.activation_1),
            
            # Dense(units=128, activation= config.activation_1),
            # BatchNormalization(),
#             LeakyReLU(alpha=0.2),
#             Dropout(rate=config.dropout_2),
            
            Dense(units=3, activation=config.activation_2)
        ])

        # Compile the model
        model.compile(
            loss=config.loss,
            optimizer=Adam(learning_rate=config.learning_rate),
            metrics=[config.metric_Acc, 
                     TopKCategoricalAccuracy(k= 2, name='Top2_Acc'), 
                     AUC(multi_label= True, num_labels= 3), 
                     Precision(thresholds=0.6), 
                     Recall(thresholds=0.2),
                     TruePositives(), 
                     FalsePositives(), 
                     TrueNegatives(), 
                     FalseNegatives()
                    ])

        return model
    rad_model = build_transfer_learning_model(
    base_model = InceptionResNetV2(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False)
    )
    rad_model.summary()
    tf.keras.utils.plot_model(rad_model, to_file=f"{CODE_OUTPUTS}rad_model+top.png", dpi= 300,  show_shapes=True)

#     rad_model.summary()
    callbacks = [WandbMetricsLogger("epoch"),
#                 WandbModelCheckpoint("models"),
                WandbCallback(log_evaluation= True, save_model= True),
                PRMetrics(valid_data, num_log_batches= 32), 
                csv_logger
                ]
#                               save_model= True,
    rad_hist = rad_model.fit(
    train_data,
    validation_data= valid_data,
    steps_per_epoch= n_samples_train// config.batch_size,
    validation_steps= n_samples_val// config.batch_size,
    epochs=config.epoch,
    callbacks= callbacks
    )
    wandb.finish()
    return rad_model, rad_hist, valid_data, test_data
#     return n_samples_train, n_samples_val, n_samples_test
    

In [21]:
class PRMetrics(Callback):
  """ Custom callback to compute per-class PR & ROC curves
  at the end of each training epoch"""
  def __init__(self, generator=None, num_log_batches=1):
    self.generator = generator
    self.num_batches = num_log_batches
    self.val_f1s = []
    self.val_recalls = []
    self.val_precisions = []
    # store full names of classes
#     self.class_names = { v: k for k, v in generator.next() }
#     self.flat_class_names = [k for k, v in generator.next()]
    self.class_names= {0: 'glioma', 1: 'meningioma', 2: 'pituitary'}
    self.flat_class_names = ['glioma', 'meningioma', 'pituitary']

  def on_epoch_end(self, epoch, logs={}):

    val_data, val_labels = self.generator.next()
    # print("val_data: ", val_data.shape)
    # print("val_labels: ", val_labels.shape)
    # use the trained model to generate predictions for the given number
    # of validation data batches (num_batches)
    val_predictions = self.model.predict(val_data)
    print(val_predictions.shape)
#     val_prob = self.model.predict_proba(val_data)
#     print(val_prob.shape)
    ground_truth_class_ids = val_labels.argmax(axis=1)
    
#      val_predict = (np.asarray(self.model.predict(self.model.validation_data[0]))).round()
#     val_targ = val_labels
#     _val_f1 = f1_score(val_targ, val_predictions)
#     _val_recall = recall_score(val_targ, val_predictions)
#     _val_precision = precision_score(val_targ, val_predictions)
#     self.val_f1s.append(_val_f1)
#     self.val_recalls.append(_val_recall)
#     self.val_precisions.append(_val_precision)
#     print(f"_val_f1: {_val_f1},  _val_precision: { _val_precision}, _val_recall: {_val_recall}")
    
    


    # Log precision-recall curve
    # the key "pr_curve" is the id of the plot--do not change
    # this if you want subsequent runs to show up on the same plot
    wandb.log({"roc_curve" : wandb.plot.roc_curve(ground_truth_class_ids, val_predictions, labels= self.flat_class_names)})
#     wandb.log({"confusion_matrix" : wandb.sklearn.plot_confusion_matrix(ground_truth_class_ids, val_predictions, labels= self.flat_class_names)})

    
#     wandb.sklearn.plot_classifier(clf, X_train, X_test, y_train, y_test, y_pred, y_probas, labels= self.flat_class_names, model_name='rad_model', feature_names=None)
    
#     wandb.log({"conf_mat" : wandb.plot.confusion_matrix(probs=None, y_true=ground_truth_class_ids, preds=val_predictions, class_names=self.flat_class_names)})


In [22]:
rad_model, rad_hist, valid_data, test_data = train()

# test_data, test_labels = test_data
# print("test_data: ", test_data.shape)
# print("test_labels: ", test_labels.shape)

# Y_pred = rad_model.predict(test_data,  batch_size=16)


# y_pred = np.argmax(Y_pred, axis=1)
# print('Confusion Matrix')


# print(confusion_matrix(test_labels, y_pred))
# print('Classification Report')

# target_names = ['glioma', 'meningioma', 'pituitary']
# print(classification_report(test_labels, y_pred, target_names=target_names))



n_samples_train:  2656
n_samples_val:  512
n_samples_test:  288
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 inception_resnet_v2 (Functi  (None, 2, 2, 1536)       54336736  
 onal)                                                           
                                                                 
 global_average_pooling2d (G  (None, 1536)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               196736    
                                                                 
 dense_1 (Dense)             (None, 3)                 387       
                                                                 
Total params: 54,533,859
Trainable params: 197,123
Non-trainable params: 54,336,736
________________________________________

wandb: WARNING The save_model argument by default saves the model in the HDF5 format that cannot save custom objects like subclassed models and custom layers. This behavior will be deprecated in a future release in favor of the SavedModel format. Meanwhile, the HDF5 model is saved as W&B files and the SavedModel as W&B Artifacts.
wandb: WARNING WandbCallback is unable to log validation data. When using a generator for validation_data, you must pass validation_steps


Epoch 1/50
83/83 [==============================] - ETA: 0s - loss: 0.8363 - accuracy: 0.6139 - Top2_Acc: 0.8776 - auc: 0.7917 - precision: 0.7402 - recall: 0.8863 - true_positives: 1355.0000 - false_positives: 657.0000 - true_negatives: 4653.0000 - false_negatives: 1300.0000

wandb: Adding directory to artifact (/content/wandb/run-20230317_225544-97rhwde2/files/model-best)... Done. 1.1s


1/1 [==============================] - 4s 4s/step
(32, 3)
83/83 [==============================] - 124s 1s/step - loss: 0.8363 - accuracy: 0.6139 - Top2_Acc: 0.8776 - auc: 0.7917 - precision: 0.7402 - recall: 0.8863 - true_positives: 1355.0000 - false_positives: 657.0000 - true_negatives: 4653.0000 - false_negatives: 1300.0000 - val_loss: 0.6750 - val_accuracy: 0.6895 - val_Top2_Acc: 0.9180 - val_auc: 0.8659 - val_precision: 0.8389 - val_recall: 0.9336 - val_true_positives: 311.0000 - val_false_positives: 91.0000 - val_true_negatives: 933.0000 - val_false_negatives: 201.0000
Epoch 2/50
83/83 [==============================] - ETA: 0s - loss: 0.6333 - accuracy: 0.7292 - Top2_Acc: 0.9416 - auc: 0.8848 - precision: 0.8352 - recall: 0.9330 - true_positives: 1759.0000 - false_positives: 522.0000 - true_negatives: 4788.0000 - false_negatives: 896.0000

...

wandb: Adding directory to artifact (/content/wandb/run-20230317_225544-97rhwde2/files/model-best)... Done. 1.1s


1/1 [==============================] - 0s 62ms/step
(32, 3)
83/83 [==============================] - 103s 1s/step - loss: 0.2399 - accuracy: 0.9085 - Top2_Acc: 0.9891 - auc: 0.9840 - precision: 0.9446 - recall: 0.9748 - true_positives: 2388.0000 - false_positives: 216.0000 - true_negatives: 5094.0000 - false_negatives: 267.0000 - val_loss: 0.3729 - val_accuracy: 0.8320 - val_Top2_Acc: 0.9707 - val_auc: 0.9555 - val_precision: 0.8933 - val_recall: 0.9414 - val_true_positives: 425.0000 - val_false_positives: 69.0000 - val_true_negatives: 955.0000 - val_false_negatives: 87.0000
Epoch 25/50
1/1 [==============================] - 0s 107ms/step
(32, 3)
83/83 [==============================] - 8s 100ms/step - loss: 0.2366 - accuracy: 0.9153 - Top2_Acc: 0.9902 - auc: 0.9834 - precision: 0.9441 - recall: 0.9706 - true_positives: 2407.0000 - false_positives: 195.0000 - true_negatives: 5115.0000 - false_negatives: 248.0000 - val_loss: 0.4706 - val_accuracy: 0.8281 - val_Top2_Acc: 0.9648 - val_auc

wandb: Adding directory to artifact (/content/wandb/run-20230317_225544-97rhwde2/files/model-best)... Done. 1.2s


1/1 [==============================] - 0s 62ms/step
(32, 3)
83/83 [==============================] - 102s 1s/step - loss: 0.1694 - accuracy: 0.9397 - Top2_Acc: 0.9940 - auc: 0.9917 - precision: 0.9575 - recall: 0.9804 - true_positives: 2489.0000 - false_positives: 145.0000 - true_negatives: 5165.0000 - false_negatives: 166.0000 - val_loss: 0.3723 - val_accuracy: 0.8613 - val_Top2_Acc: 0.9727 - val_auc: 0.9576 - val_precision: 0.8853 - val_recall: 0.9316 - val_true_positives: 431.0000 - val_false_positives: 67.0000 - val_true_negatives: 957.0000 - val_false_negatives: 81.0000
Epoch 39/50
1/1 [==============================] - 0s 66ms/step
(32, 3)
83/83 [==============================] - 8s 92ms/step - loss: 0.1755 - accuracy: 0.9337 - Top2_Acc: 0.9940 - auc: 0.9905 - precision: 0.9566 - recall: 0.9744 - true_positives: 2467.0000 - false_positives: 161.0000 - true_negatives: 5149.0000 - false_negatives: 188.0000 - val_loss: 0.5603 - val_accuracy: 0.7988 - val_Top2_Acc: 0.9727 - val_auc: 

Top2_Acc,▁▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇▇▇█▇█▇█████████████
accuracy,▁▃▃▄▅▅▅▅▅▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████
auc,▁▄▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇▇▇███▇██▇██████████
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch/Top2_Acc,▁▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇▇▇█▇█▇█████████████
epoch/accuracy,▁▃▃▄▅▅▅▅▅▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████
epoch/auc,▁▄▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇▇▇███▇██▇██████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch/false_negatives,█▆▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁
epoch/false_positives,█▆▇▆▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▃▃▃▂▃▂▂▂▂▂▂▂▁▂▁
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁


In [23]:
# print(rad_hist.history)
# del rad

In [24]:
# import scikitplot as skplt
# rad_y_proba = rad_model.predict(test_data)
# print(rad_y_proba)
# plot = skplt.metrics.plot_roc(test_data, rad_y_proba)
# plt.title("ROC Curves - K-Nearest Neighbors")
# plt.show()
# # wandb.agent(sweep_id, train, count=8)

In [ ]:
from sklearn.metrics import accuracy_score

all_test_data = []
all_test_labels = []
for i in range(257):
  test_images, test_labels = test_data.next()
  all_test_data.append(test_images)
  all_test_labels.append(test_labels)


Y_pred = rad_model.predict(all_test_data)

y_pred = np.argmax(Y_pred, axis=1)
print('Confusion Matrix')

# y_val=np.argmax(y_val, axis=1)


print(confusion_matrix(all_test_labels, y_pred))
print('Classification Report')

target_names = ['glioma', 'meningioma', 'pituitary']
print(classification_report(all_test_labels, y_pred, target_names=target_names))

accuracy = accuracy_score(all_test_labels, y_pred)
print(f"The Accuracy is: {accuracy}")

In [32]:
# !pip install scikit-plot

# import scikitplot as skplt

# plot = skplt.metrics.plot_roc(test_data, Y_pred)
# plt.title("ROC Curves - K-Nearest Neighbors")
# plt.show()
# # rad_model.summary()

In [ ]:
# from tensorflow.keras.utils import plot_model
# plot_model(rad_model, to_file='/kaggle/working/model.png')

In [ ]:

# path = "/kaggle/working/"
# saved_model = path + "model_Kaggle_03_84" + "_" + str(config.dropout_1)+ "_" +str(config.dropout_2) + "_"+ str(config.learning_rate) + "_" + str(config.batch_size) + "_" + config.optimizer + ".h5"
# print(saved_model)
# print("Saving: ", saved_model)
# rad_model.save(saved_model)

In [ ]:
# print("Evaluate on test data")

# results = rad_model.evaluate(test_data, batch_size=32)
# print("test loss, test acc:", results)

In [ ]:
# Y_pred = rad_model.predict(test_data, 480 // 32+1)
# y_pred = np.argmax(Y_pred, axis=1)
# print('Confusion Matrix')
# print(confusion_matrix(test_data.classes, y_pred))
# print('Classification Report')
# target_names = ['glioma', 'meningioma', 'pituitary']
# print(classification_report(test_data.classes, y_pred, target_names=target_names))

In [ ]:
# plt.plot(rad_hist.history['loss'])
# plt.plot(rad_hist.history['val_loss'])
# plt.title('model loss')
# plt.ylabel('loss')
# plt.xlabel('epoch')
# plt.legend(['train', 'validation'], loc='upper right')
# plt.show()

In [ ]:
# print(rad_hist.history.keys())
# # summarize history for accuracy
# plt.plot(rad_hist.history['accuracy'])
# plt.plot(rad_hist.history['val_accuracy'])
# plt.title('model accuracy')
# plt.ylabel('accuracy')
# plt.xlabel('epoch')
# plt.legend(['train', 'validation'], loc='lower right')
# plt.show()